# 01 · Problem framing, data audit and cleaning
**Data Analytics Competition 2026 · Conflict analytics: Russia–Ukraine, Israel–Hamas, Afghanistan–Pakistan, US–Iran**

### The question we chose
> **When and where will conflict deaths rise next, and what does that mean for the Indian Armed Forces and for humanitarian planning?**

We break this down into five sub-questions that the rest of the notebooks answer:

| # | Question | Notebook | Why a commander cares |
|---|---|---|---|
| Q1 | How intense is each conflict, and how is it changing (tempo, lethality, geography, seasonality)? | 03 | Threat picture |
| Q2 | How are the wars being fought: attrition by side, type of violence, front-line shift, cross-border spread? | 03 | Lessons for force design |
| Q3 | Can we forecast monthly deaths 1–9 months ahead better than simple rules, on data the model has never seen (Jan–Sep 2026)? | 04 | Planning horizon |
| Q4 | Can we warn, a month ahead, which sub-national areas will escalate? | 05 | Indications & warning (I&W) |
| Q5 | What does all this mean for India: western front, Gulf diaspora and energy, Indian Ocean, defence supply, refugees? | 03, report | Policy relevance |

### Data
Official competition package only (UCDP GED 26.1 + Candidate 26.01.26.09 events, UNHCR, IDMC, UNRWA, World Bank).
The top-level files in the parent folder belong to a different package and are **not** used.

This notebook loads the six tables, runs **36 documented checks**, fixes what can be fixed, flags what cannot,
and writes the cleaned tables to `data/processed/`. Source files are never modified.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
from src import config as C, viz

## 1. Raw tables at a glance

In [2]:
raw = {n: pd.read_csv(C.RAW_DIR / f"{n}.csv", encoding="utf-8-sig") for n in
       ["conflict_events", "event_sources", "bilateral_displacement_stocks", "unhcr_demographics",
        "displacement_reference_stocks", "country_year_context"]}
pd.DataFrame({n: {"rows": len(d), "columns": d.shape[1], "missing cells": int(d.isna().sum().sum())}
              for n, d in raw.items()}).T

,rows,columns,missing cells
conflict_events,61055,34,3081
event_sources,85406,4,0
bilateral_displacement_stocks,6273,8,6269
unhcr_demographics,19830,12,0
displacement_reference_stocks,112,5,0
country_year_context,1764,6,160


In [3]:
ev = raw["conflict_events"]
print("Events by case and year")
pd.crosstab(ev.conflict_case, ev.event_year, margins=True)

Events by case and year


event_year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,All
conflict_case,,,,,,,,,,,,,,
Afghanistan–Pakistan,315,322,382,504,846,291,155,322,433,405,625,714,671,5985
Israel–Hamas,919,13,12,9,51,45,8,107,47,1412,3136,3186,675,9620
Russia–Ukraine,1298,871,514,216,153,199,146,167,7534,10807,11803,8034,2813,44555
US–Iran,0,0,0,2,9,16,41,24,21,40,131,126,485,895
All,2532,1206,908,731,1059,551,350,620,8035,12664,15695,12060,4644,61055


In [4]:
print("Missing values in conflict_events:"); print(ev.isna().sum()[ev.isna().sum() > 0])
ev[["deaths_best", "deaths_low", "deaths_high", "source_count"]].describe(percentiles=[.5, .9, .99, .999]).T

Missing values in conflict_events:
location_iso3      38
admin1_name      3043
dtype: int64


,count,mean,std,min,50%,90%,99%,99.9%,max
deaths_best,"61,055.000",9.080,156.131,0.000,2.000,7.000,60.000,"1,224.542","15,996.000"
deaths_low,"61,055.000",7.974,146.417,0.000,1.000,6.000,54.000,"1,095.974","12,066.000"
deaths_high,"61,055.000",12.646,360.087,0.000,2.000,13.000,126.000,844.136,"64,183.000"
source_count,"61,055.000",1.399,1.347,1.000,1.000,2.000,6.000,14.000,149.000


## 2. Run the cleaning stage
`src/cleaning.py` implements every rule below. Key decisions:

* **Out-of-order estimates** (low > best or best > high, 692 rows): raw values kept; corrected band
  `deaths_low_c = min(low, best, high)`, `deaths_high_c = max(...)`. We widen the band rather than move `best`,
  because `best` reconciles exactly with the four component fields (validated).
* **Multi-month events** (731 rows, incl. front-wide monthly aggregates): deaths are spread over calendar months
  in proportion to the days in the date range, instead of stacking them on the start month.
* **Provisional 2026 Candidate data**: flagged and used *only* as the withheld test set.
* **FATA → Khyber Pakhtunkhwa**: Pakistan merged FATA into KP in 2018; without harmonising, the sub-national panel
  shows a fake collapse in FATA and a fake surge in KP.
* **Publisher names**: 'ITAR-TASS'/'Tass', 'UA Losses'/'UALosses', 'Source:' prefixes etc. normalised.
* **Missing geography**: blank `location_iso3` → `INTL` (waters/airspace) or `XXX` (ambiguous code 666); blank admin1 →
  explicit "country-level" bucket so the deaths are not lost.
* **World Bank gaps**: interpolated within economy; used with a one-year lag so 2026 needs no extrapolation.

In [5]:
from src import cleaning
out = cleaning.run()
dq = out["dq_log"]
print(f"{len(dq)} checks logged; {int((dq.rows_affected > 0).sum())} found issues")
dq[["id", "table", "check", "category", "rows_affected", "pct_affected", "action"]]

36 checks logged; 23 found issues


,id,table,check,category,rows_affected,pct_affected,action
0,DQ01,conflict_events,File encoding: UTF-8 byte-order mark on header,Format,0,0.000,Read with utf-8-sig so the first column is 'ev...
1,DQ02,conflict_events,Duplicate primary key (event_id),Integrity,0,0.000,Validated: none found.
2,DQ03,conflict_events,deaths_best equals sum of four component fields,Integrity,0,0.000,Validated: identity holds for every row.
3,DQ04,conflict_events,Negative fatality values,Validity,0,0.000,Validated: none found.
4,DQ05,conflict_events,Out-of-order low/best/high estimates (low<=bes...,Validity,692,1.130,Raw values preserved. Corrected copies deaths_...
5,DQ06,conflict_events,date_end earlier than date_start,Validity,0,0.000,Validated: none found.
6,DQ07,conflict_events,Event date range crosses a month boundary,Temporal,731,1.200,Fatalities allocated across months in proporti...
7,DQ08,conflict_events,Date range extends beyond the 30 Sep 2026 extr...,Temporal,1,0.000,Kept. Only the in-window share of deaths enter...
8,DQ09,conflict_events,Low date precision (code 4-5: month or longer),Precision,792,1.300,Kept and flagged. Used as a feature (share of ...
9,DQ10,conflict_events,Provisional UCDP Candidate records (Jan-Sep 2026),Provenance,4644,7.610,Flagged is_provisional. 2026 is used ONLY as t...


In [6]:
viz.fig_dq_summary(dq)

## 3. Spot-checks of the fixes

In [7]:
e = out["events"]
print("Out-of-order examples (raw vs corrected):")
e.loc[e.flag_estimate_order, ["event_id", "deaths_low", "deaths_best", "deaths_high", "deaths_low_c", "deaths_high_c"]].head()

Out-of-order examples (raw vs corrected):


,event_id,deaths_low,deaths_best,deaths_high,deaths_low_c,deaths_high_c
149,153107,8,8,7,7,8
242,153684,10,10,9,9,10
1506,154253,3,3,2,2,3
1539,150079,7,12,10,7,12
2121,154985,6,7,6,6,7


In [8]:
al = out["alloc"]
x = e.loc[e.flag_multi_month].nlargest(1, "deaths_best")[["event_id", "date_start", "date_end", "deaths_best"]]
print(x.to_string(index=False))
al[al.event_id == x.event_id.iloc[0]].assign(deaths=lambda d: d.share * x.deaths_best.iloc[0])

 event_id date_start   date_end  deaths_best
   438282 2022-03-02 2022-05-07        15996


,event_id,month,share,deaths
60927,438282,2022-03-01,0.448,"7,162.388"
60928,438282,2022-04-01,0.448,"7,162.388"
60929,438282,2022-05-01,0.104,"1,671.224"


In [9]:
# allocation conserves deaths (except the logged October share after the window)
alloc_tot = al.merge(e[["event_id", "deaths_best"]]).eval("share*deaths_best").sum()
print(f"events total {e.deaths_best.sum():,.1f}  vs  allocated total {alloc_tot:,.1f}")

events total 554,396.0  vs  allocated total 554,395.5


In [10]:
s = out["sources"]
print(f"{s.publisher.nunique():,} raw publisher labels -> {s.publisher_norm.nunique():,} normalised outlets")
s.groupby("publisher_norm").size().nlargest(15).rename("references").to_frame()

1,714 raw publisher labels -> 1,624 normalised outlets


,references
publisher_norm,
TASS,13905
Ukrinform,12689
UA Losses,10135
Wafa,6529
Russo-Ukrainian Warspotting,6032
MemoryBook,2814
Agence France Presse,2315
SATP,2212
OCHA,1967


## 4. A finding from cleaning itself: the *fog of war* in the data
Most Russia–Ukraine deaths since 2024 are recorded as country-level aggregates (often memorial-database counts),
not as located incidents. By 2026, **over 90%** of RU–UA deaths have no sub-national location.
Consequence for our method: front-line geography is measured with **located, discrete events**, not deaths, and
sub-national forecasts are evaluated with that limit stated.

In [11]:
from src import insights
gp = insights.geo_precision_trend(e)
display(gp[gp.conflict_case == "Russia–Ukraine"].round(3))
viz.fig_fog_of_war(gp)

,conflict_case,year,events,deaths,share_deaths_country_level,share_deaths_aggregate,uncertainty_index
26,Russia–Ukraine,2014,1298,4859,0.199,0.369,0.175
27,Russia–Ukraine,2015,871,1894,0.228,0.228,0.112
28,Russia–Ukraine,2016,514,699,0.030,0.069,0.411
29,Russia–Ukraine,2017,216,447,0.036,0.512,0.168
30,Russia–Ukraine,2018,153,243,0.350,0.255,0.387
31,Russia–Ukraine,2019,199,311,0.132,0.408,0.167
32,Russia–Ukraine,2020,146,197,0.056,0.299,0.736
33,Russia–Ukraine,2021,167,208,0.130,0.240,0.226
34,Russia–Ukraine,2022,7534,100671,0.299,0.660,1.135
35,Russia–Ukraine,2023,10807,76567,0.374,0.432,1.469


## 5. Summary
* 36 checks, 0 key or arithmetic failures, 692 out-of-order estimate rows repaired (flagged),
  731 multi-month records disaggregated, 4,644 provisional rows isolated for testing.
* Cleaned tables are in `data/processed/*.parquet`; the full log is `outputs/tables/data_quality_log.csv`.

Next: `02_feature_engineering.ipynb`.